# Análisis de la morfología interna del fruto en Cranberry – Estampas

En este tutorial, demostraremos cómo realizar el análisis de la morfología interna de frutos de cranberry usando imágenes de estampas mediante `FruitInternalAnalyzer`. Aquí nos enfocaremos en qué parámetros ajustar cuando tenemos este tipo de imágenes. Para revisar qué hace cada método con más detalle y cómo se realiza un análisis completo con `FruitInternalAnalyzer`, ver el tutorial de [Análisis de morfología y color en Cranberry](https://traitly.readthedocs.io/en/latest/es/tutorials/cranberry_internal_analysis/).

El primer paso es inicializar la clase `FruitInternalAnalyzer` y cargar nuestra imagen.

In [ ]:
# Importar la clase
from traitly.fruit_phenotyping import FruitInternalAnalyzer

In [ ]:
# Crear el objeto `cranberry`
path = "./cranberry_stamps.jpg"
cranberry = FruitInternalAnalyzer(path)


# Cargar nuestra imagen
cranberry.load_image()

Al momento de realizar las estampas, puede ser que algunas de ellas tengan errores que dificulten el análisis. Hay varias maneras de eliminar esos frutos de la imagen: una de ellas sería eliminar manualmente de la máscara las estampas con `cranberry.edit_mask()`, o más sencillamente, marcar con una X nuestra estampa. Esto se hace con la intención de filtrar posteriormente estos frutos según su circularidad en `cranberry.detect_fruits`. En este ejemplo, seguimos esta última estrategia.

A pesar de que la imagen no incluye ninguna referencia de tamaño, aún podemos convertir píxeles a centímetros si conocemos el tamaño de la hoja escaneada en centímetros. Las mediciones se pasan como se muestra a continuación:

In [ ]:
cranberry.setup_measurements(width_cm = 21.6, 
                             length_cm = 27.9)

Ahora procederemos a crear la máscara binaria, en donde separaremos el fondo de los demás objetos en la imagen. Por default, `generate_fruit_mask` espera un fondo de color negro, pero en las imágenes de estampas el fondo es de color blanco. En este caso, utilizaremos el parámetro `stamp=True`, lo que le indica a `generate_fruit_mask` que debe invertir el color de la imagen antes de generar la máscara, de modo que el color blanco del papel pase a ser negro y la máscara pueda segmentar correctamente el fondo de las estampas.


In [ ]:
cranberry.generate_fruit_mask(stamp = True)

Con la máscara lista, podemos proceder como con cualquier otro tipo de imagen, detectando frutos con `detect_fruits`. Si las estampas contienen pequeños espacios sin tinta, estos podrían causar ruido en la detección de los lóculos, para lo cual podemos ajustar `min_locule_area`.

Como podemos ver en los resultados, las estampas marcadas con una cruz no fueron detectadas como frutos, gracias a que filtramos contornos según su circularidad con `min_fruit_circularity=0.5`.


In [ ]:
cranberry.detect_fruits(plot = True, 
                        plot_size = (15,15),
                        min_locule_area = 300)

Con los frutos detectados, realizamos el análisis morfológico con `analyze_morphology`.

En ocasiones, los contornos de las estampas no se encuentran bien definidos, lo que puede impactar en los resultados de morfología subestimando mediciones como el área, circularidad, perímetro, etc. Vemos un ejemplo más detallado en el fruto 15.


In [ ]:
# Analizar morfologia de los frutos
cranberry.analyze_morphology(display_table = False, plot_size = (15,15))

# Visualizar estampa num. 15
cranberry.generate_single_fruit_masks(fruit_id = 15)

Para este tipo de situaciones, en lugar de utilizar el contorno original de la estampa (`contour_mode='raw'`), podemos aplicar una transformación con `contour_mode='hull'`, el cual aplica un convex hull con ayuda de la librería [cv2](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html#ga014b28e56cb8854c0de4a211cb2be656), corrigiendo las hendiduras de las estampas. Este método toma todos los puntos que definen el contorno original de un fruto y busca un nuevo contorno que los envuelva de manera convexa. Puedes pensarlo como si ajustaras una liga alrededor del fruto, "rellenando" los huecos en el perímetro de la estampa, tal como se puede ver en detalle en la estampa 15 revisada previamente.


In [ ]:
cranberry.analyze_morphology(display_table = False,
                             contour_mode = 'hull', 
                             plot_size = (15,15))

¡Con esto concluímos el análisis! así que procedemos a guardar los resultados.

In [ ]:
cranberry.results.save_all()